# Kaggle GPU embed keyframes — SigLIP 2 so400m-384 (ứng viên A1)

Encode toàn bộ keyframe của **873 video** (CÙNG bộ keyframe dataset đã dùng cho BEiT-3, xem
`notebooks/kaggle_embed_keyframes.ipynb`) bằng SigLIP 2
(`google/siglip2-so400m-patch14-384`, `SigLip2Encoder` ở `app/encoders/siglip2.py`), xuất
**artifact vector** ra `/kaggle/working/embed_out_siglip2/`. Đây là candidate A1 của
`.claude/plans/encoder-swap-siglip2-pecore.plan.md` §T6.

**Notebook này KHÔNG ingest vào Milvus/MongoDB nào cả** — cùng nguyên tắc phân vai với
notebook BEiT-3: *Kaggle chỉ làm việc cần GPU hoặc cần ảnh, local giữ toàn bộ sự thật về
metadata*. Import artifact này vào DB local (qua cờ `--encoder siglip2` của
`scripts/import_embed_out.py`, T5 của plan lớn) là bước riêng, không nằm trong notebook này.

## Đơn giản hơn notebook BEiT-3 ở điểm nào

- **Không cần dựng môi trường conda Python 3.10 riêng.** `app/encoders/__init__.py` rỗng,
  nên `import app.encoders.siglip2` không kéo theo `timm==0.4.12`/`torchscale` mà BEiT-3
  cần — chạy thẳng trên kernel mặc định của Kaggle.
- **Không cần script tải checkpoint riêng.** `SigLip2Encoder` tải weight qua
  `transformers.AutoModel.from_pretrained(..., revision=...)` từ HuggingFace Hub, tự cache.
- Đổi lại: phải pin `transformers>=4.49,<5.0` — bản `>=5.0` tự nâng yêu cầu lên
  `torch>=2.5` và ÂM THẦM tắt PyTorch nếu torch cài sẵn thấp hơn (bug có thật, gặp lúc test
  local 2026-08-13, xem cell mục 3). KHÔNG tự ý cài đè `torch` — dùng bản Kaggle đã cài sẵn
  (đã có CUDA khớp driver của instance).

## Đầu ra

```
embed_out_siglip2/
├── manifest.json                  # manifest v2: model_id, hf_revision, weight_files_sha256, ...
├── probe/
│   ├── probe_texts.json            # 20 câu cố định (giống hệt bộ đã dùng cho BEiT-3)
│   ├── probe_text_vectors.npy      # (20, 1152) float32
│   ├── image_probe.json            # 16 keyframe_id cố định, chọn trải đều qua các batch
│   └── image_probe_vectors.npy     # (16, 1152) float32 -- MỚI so với artifact BEiT-3,
│                                    # khả thi vì local giờ đã có ảnh (T4 của plan lớn)
└── vectors/
    ├── L29_V012.npy                 # (n_kf, 1152) float32, ĐÃ L2-normalize
    ├── L29_V012.jsonl               # n_kf dòng, THỨ TỰ KHỚP HÀNG của .npy
    └── ...
```

`keyframe_id = "{video_id}_f{frame_idx}"` — CÙNG hợp đồng với artifact BEiT-3
(`.claude/plans/kaggle-offline-embed.plan.md` §3), nên phía import không cần biết artifact
này tới từ encoder nào để đọc đúng.

## Thứ tự chạy

1. Kiểm tra GPU (ghi lại T4 hay P100 — ảnh hưởng throughput và độ trôi số fp16)
2. Clone repo qua Kaggle Secret `GITHUB_TOKEN`
3. Cài `transformers>=4.49,<5.0` (kernel mặc định, không env riêng)
4. Cấu hình — mọi hằng số dồn về đúng một cell
5. Kiểm kê bộ keyframe — fail loud nếu lệch
6. Sinh script embed ra `/kaggle/working/embed_siglip2.py`
7. **Smoke test ~1.000 keyframe** — đo throughput thật trước khi treo job dài
8. **Chạy full 873 video** — chỉ bấm sau khi mục 7 xanh
9. Kiểm tra output
10. Image probe + text probe
11. Sinh `manifest.json` (v2)
12. Hướng dẫn mang về local

## Cảnh báo tốn tài nguyên & ngân sách GPU

Mục 8 là cell nặng nhất. **Chưa có số đo throughput thật cho SigLIP2 so400m@384** — so400m
nặng hơn `beit3_base@224` đáng kể (400M tham số vision tower vs ~220M, ảnh 384px vs 224px),
nên KHÔNG dùng lại ước lượng ~1 giờ của notebook BEiT-3. Mục 7 (smoke test) cho số đo thật
đầu tiên. Kaggle giới hạn **30 giờ GPU/tuần** — nếu cộng cả candidate A2 (PE-Core) vượt
ngân sách này, dừng lại và ưu tiên chạy xong A1 trước.

## Việc cần làm TRƯỚC khi mở notebook này trên Kaggle

1. **Commit + push nhánh chứa code này lên GitHub** — `app/encoders/siglip2.py`,
   `configs/model/siglip2.yaml` phải có mặt trên nhánh mà biến `REPO_BRANCH` (mục 2) trỏ
   tới. Hiện các file này CHƯA được commit — đổi `REPO_BRANCH` nếu bạn không dùng `main`.
2. **Kaggle Secret `GITHUB_TOKEN`** (repo `tiendze/AIC-Video-Retrieval` là private).
3. **Add Kaggle Dataset keyframe** — CÙNG dataset đã dùng cho notebook BEiT-3, cấu trúc
   `<batch>/<video_id>/<video_id>_<frame_idx>.jpg`.
4. **Bật Internet + GPU accelerator** trong Notebook Settings.

## Ngoài phạm vi notebook này

- **Candidate A2 (PE-Core-bigG-14-448)** — notebook riêng `kaggle_embed_pecore.ipynb`
  (chưa viết), dependency xung khắc với candidate này (`timm>=1.0.19` vs không cần timm ở
  đây).
- **Import vào Milvus/MongoDB local** — qua `scripts/import_embed_out.py --encoder
  siglip2` (T5 của plan lớn, **chưa implement** tại thời điểm viết notebook này).


## 1. Kiểm tra GPU

Ghi lại tên GPU hiện ra ở dòng `nvidia-smi` (T4 hay P100) — ảnh hưởng cả throughput lẫn độ trôi số fp16 giữa Kaggle và RTX 3070 local (plan §T4, ngưỡng image probe).

In [ ]:
!nvidia-smi

## 2. Clone repo qua Kaggle Secret `GITHUB_TOKEN`

Yêu cầu: đã thêm Kaggle Secret tên `GITHUB_TOKEN` (Add-ons → Secrets) chứa Personal Access
Token có quyền đọc repo `tiendze/AIC-Video-Retrieval` (private). Clone không phải để lấy
code mới — mà vì cần `app/encoders/siglip2.py` đã có sẵn và chạy được (test hợp đồng đã
xanh ở local 2026-08-13), chép lại vào cell là vô nghĩa.

In [ ]:
from pathlib import Path

from kaggle_secrets import UserSecretsClient

github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")

REPO_BRANCH = "main"  # ĐỔI nếu bạn commit code này lên nhánh khác -- phải PUSH lên GitHub
                       # trước khi mở notebook này (app/encoders/siglip2.py, configs/model/
                       # siglip2.yaml phải có mặt trên nhánh này)
REPO_DIR = Path("/kaggle/working/AIC-Video-Retrieval")
if not REPO_DIR.exists():
    clone_url = f"https://{github_token}@github.com/tiendze/AIC-Video-Retrieval.git"
    !git clone --branch {REPO_BRANCH} {clone_url} {REPO_DIR}
else:
    print(f"{REPO_DIR} đã tồn tại, bỏ qua clone (session được resume).")

%cd {REPO_DIR}
!git log --oneline -3


## 3. Cài dependency (kernel mặc định — KHÔNG cần env Python 3.10 riêng)

`SigLip2Encoder` chỉ cần `torch` (đã có sẵn trên Kaggle, KHÔNG cài đè — bản Kaggle đã khớp
CUDA driver của instance) + `transformers` (cài mới) + `Pillow` (thường có sẵn). Không
`pip install -e .` — `pyproject.toml` khai `requires-python = ">=3.10,<3.11"`, kernel mặc
định Kaggle là 3.11/3.12 nên pip sẽ từ chối; chỉ cần thêm repo vào `sys.path`.

⚠️ **Bẫy shadow module**: repo có package `datasets/` trùng tên thư viện HF `datasets`.
Script embed ở mục 6 dùng `sys.path.append(...)` (thêm vào **cuối**), không
`sys.path.insert(0, ...)` — nếu chèn ở đầu, `import datasets` (nếu bất kỳ thư viện nào
trong chuỗi phụ thuộc gọi tới) sẽ bắt nhầm package của repo.

In [ ]:
!pip install -q "transformers>=4.49,<5.0"

import torch
import transformers
from transformers.utils.import_utils import is_torch_available

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

if not is_torch_available():
    raise RuntimeError(
        "transformers không thấy PyTorch dù `import torch` chạy được -- ĐÂY LÀ ĐÚNG BẪY đã "
        "gặp lúc test local 2026-08-13: transformers>=5.0 tự nâng yêu cầu lên torch>=2.5 và "
        "âm thầm tắt AutoModel nếu torch cài sẵn thấp hơn (xem pyproject.toml comment ở "
        "extra encoder-candidates). Kiểm tra lại torch.__version__ ở trên; nếu Kaggle đổi "
        "image và torch quá cũ, đây là dấu hiệu cần báo lại, không phải lỗi cài đặt."
    )


## 4. Cấu hình

Mọi hằng số dồn về đúng một cell (notebook không tách `configs/*.yaml` — quyết định gộp đã chốt ở `.claude/plans/kaggle-offline-embed.plan.md` §4/§5, giữ nguyên cho notebook này).

In [ ]:
import json
from pathlib import Path

# Cấu trúc bắt buộc: <batch>/<video_id>/<video_id>_<frame_idx>.jpg (3 tầng) -- CÙNG bộ
# keyframe dataset đã dùng cho BEiT-3 (notebooks/kaggle_embed_keyframes.ipynb).
KF_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/embed_out_siglip2")
VECTORS_DIR = OUT_DIR / "vectors"
PROBE_DIR = OUT_DIR / "probe"

CHUNK_SIZE = 256        # số ảnh mở cùng lúc mỗi video trước khi đưa vào encoder (bộ nhớ)
BATCH_SIZE = 16         # batch GPU thật sự truyền cho SigLip2Encoder -- CỐ Ý bảo thủ so với
                        # BATCH_SIZE=32 của notebook BEiT-3: so400m@384 nặng hơn beit3_base@224
                        # đáng kể (400M tham số vision tower vs ~220M, ảnh 384px vs 224px).
                        # Đòn bẩy đầu tiên khi cần tăng throughput là PREPROCESS_WORKERS,
                        # không phải BATCH_SIZE (bài học đo được từ ClipEncoder 2026-08-12:
                        # decode ảnh CPU là nút thắt, không phải GPU) -- chỉ tăng BATCH_SIZE
                        # sau khi đã thử PREPROCESS_WORKERS và thấy GPU vẫn rảnh.
PREPROCESS_WORKERS = 8

HF_REPO_ID = "google/siglip2-so400m-patch14-384"
HF_REVISION = "e8e487298228002f3d8a82e0cd5c8ea9c567f57f"  # commit SHA ghim cứng, xem configs/model/siglip2.yaml
MODEL_ID = "siglip2_so400m_p14_384"  # phải khớp configs/model/siglip2.yaml (local)
EMBEDDING_DIM = 1152  # verified 2026-08-13 từ config.json -- chỉ dùng để cảnh báo lệch, không ép buộc

EXPECTED_NUM_VIDEOS = 873
EXPECTED_NUM_KEYFRAMES = 221_903  # cùng bộ dataset đã đo cho BEiT-3, xem kaggle-offline-embed.plan.md §2

IMAGE_PROBE_COUNT = 16  # T4 của plan lớn: giờ local đã có ảnh, probe bằng ảnh mạnh hơn probe text đơn thuần

# Giữ NGUYÊN VĂN cùng 20 câu probe text đã dùng cho BEiT-3 (kaggle_embed_keyframes.ipynb) --
# không phải vì nội dung câu quan trọng, mà để cùng một bộ câu hỏi có thể tái dùng nếu sau
# này muốn so sánh "câu nào hai encoder xử lý khác nhau nhiều nhất" một cách công bằng.
PROBE_TEXTS = [
    "một người đàn ông đang phát biểu trước ống kính",
    "cảnh quay từ trên cao xuống một con đường đông xe cộ",
    "phóng viên đứng trước tòa nhà chính phủ đưa tin",
    "hai người đang bắt tay nhau trong một sự kiện",
    "một chiếc xe cứu hỏa đang di chuyển trên đường phố",
    "đám đông tụ tập tại một quảng trường lớn",
    "một đầu bếp đang chế biến món ăn trong nhà bếp",
    "trận bóng đá đang diễn ra trên sân vận động",
    "một chiếc máy bay cất cánh từ đường băng",
    "người dẫn chương trình đang giới thiệu khách mời",
    "a man is speaking in front of a camera",
    "aerial view of a crowded street with many cars",
    "a reporter standing in front of a government building",
    "two people shaking hands at an event",
    "a fire truck moving through the street",
    "a crowd gathered in a large square",
    "a chef preparing food in a kitchen",
    "a football match taking place in a stadium",
    "an airplane taking off from a runway",
    "a host introducing a guest on a talk show",
]
assert len(PROBE_TEXTS) == 20

print("KF_ROOT:", KF_ROOT)
print("OUT_DIR:", OUT_DIR)
print("CHUNK_SIZE:", CHUNK_SIZE, "| BATCH_SIZE:", BATCH_SIZE, "| PREPROCESS_WORKERS:", PREPROCESS_WORKERS)
print("HF_REPO_ID:", HF_REPO_ID, "| HF_REVISION:", HF_REVISION)
print("MODEL_ID:", MODEL_ID)


## 5. Kiểm kê bộ keyframe (kernel mặc định, không cần GPU)

Chạy trước khi sinh/chạy script embed — nếu có file lạc thư mục, trùng
`(video_id, frame_idx)`, hay tên file không parse được, **dừng ở đây**. Cùng logic kiểm kê
với `kaggle_embed_keyframes.ipynb` mục 7 (cùng dataset, cùng regex).

In [ ]:
import re
from collections import Counter

_FILENAME_RE = re.compile(r"^(L\d+_V\d+)_(\d+)\.(jpg|jpeg|png)$", re.IGNORECASE)

image_paths = sorted(
    p for p in KF_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
)
print(f"Tìm thấy {len(image_paths)} file ảnh dưới {KF_ROOT}")
if not image_paths:
    raise FileNotFoundError(
        f"Không tìm thấy ảnh keyframe nào dưới {KF_ROOT} -- kiểm tra đã Add đúng Kaggle "
        "Dataset keyframe vào notebook chưa (CÙNG dataset đã dùng cho notebook BEiT-3)."
    )

parse_errors: list[str] = []
seen: dict[tuple[str, int], Path] = {}

for path in image_paths:
    match = _FILENAME_RE.match(path.name)
    if not match:
        parse_errors.append(f"tên file không khớp pattern: {path}")
        continue
    video_id, frame_idx_str, _ext = match.groups()
    if path.parent.name != video_id:
        parse_errors.append(
            f"video_id trong tên file ({video_id}) không khớp thư mục cha ({path.parent.name}): {path}"
        )
        continue
    key = (video_id, int(frame_idx_str))
    if key in seen:
        parse_errors.append(f"trùng (video_id, frame_idx)={key}: {path} và {seen[key]}")
        continue
    seen[key] = path

if parse_errors:
    print(f"\n{len(parse_errors)} lỗi kiểm kê -- in tối đa 50 dòng đầu:")
    for line in parse_errors[:50]:
        print(" -", line)
    raise ValueError(
        f"{len(parse_errors)} lỗi kiểm kê bộ keyframe -- dừng lại, không chạy embed trên dữ "
        "liệu chưa xác thực."
    )

video_ids = sorted({video_id for video_id, _ in seen})
per_video_counts = Counter(video_id for video_id, _ in seen)

print(f"Kiểm kê OK: {len(seen)} keyframe hợp lệ, {len(video_ids)} video duy nhất, 0 lỗi.")
print(f"Kỳ vọng: {EXPECTED_NUM_KEYFRAMES} keyframe / {EXPECTED_NUM_VIDEOS} video.")
if len(seen) != EXPECTED_NUM_KEYFRAMES or len(video_ids) != EXPECTED_NUM_VIDEOS:
    print(
        "CẢNH BÁO: số đo không khớp kỳ vọng -- dataset trên Kaggle có thể khác bản đã đo. "
        "Không raise (dữ liệu vẫn hợp lệ với chính số liệu của nó), nhưng kiểm tra lại "
        "trước khi tin manifest.json ở bước cuối."
    )

batch_counts = Counter(video_id.split("_V", 1)[0] for video_id in video_ids)
for batch, count in sorted(batch_counts.items()):
    print(f"  {batch}: {count} video")


## 6. Sinh script embed (`%%writefile`)

Cùng lý do như notebook BEiT-3: code embed cần chạy như script riêng (import
`app.encoders.siglip2` qua `sys.path`, không phải qua magic cell). Script tự đảm bảo:

- **Sort tất định** theo `(video_id, frame_idx)` — giữ nguyên `τ < t` của DANTE.
- **`keyframe_id = "{video_id}_f{frame_idx}"`** — CÙNG hợp đồng với artifact BEiT-3.
- **Resume theo video**, ghi file tạm rồi `os.replace` — không bao giờ có `.npy`/`.jsonl`
  viết dở mà trông như đã xong.
- **`--limit-keyframes`** (không chỉ `--limit-videos`): mật độ keyframe/video lệch rất
  nhiều giữa các batch (trung vị 90 ở L23 tới 689 ở L22, `kaggle-offline-embed.plan.md`
  §2) — smoke test theo SỐ KEYFRAME cho ước lượng công bằng hơn theo số video.
- **`sys.path.append`** (thêm vào cuối), không `insert(0, ...)` — né bẫy shadow module
  `datasets/` đã nói ở mục 3.

In [ ]:
%%writefile /kaggle/working/embed_siglip2.py
"""Embed AIC 2026 keyframes with SigLIP 2 (so400m-384) on Kaggle GPU -- candidate A1
of the encoder-swap plan (.claude/plans/encoder-swap-siglip2-pecore.plan.md T6).

Reads pre-extracted keyframes from a Kaggle Dataset (<batch>/<video_id>/<video_id>_<frame_idx>.jpg)
-- the SAME dataset already used for the BEiT-3 embed run -- encodes each image with
SigLip2Encoder (app/encoders/siglip2.py, cloned from the repo, imported via sys.path, NOT
installed as a package), and writes one (.npy, .jsonl) pair per video_id under
--out-dir/vectors/. Row i of the .jsonl always corresponds to row i of the .npy, and
keyframe_id is "{video_id}_f{frame_idx}" -- the SAME sort/id contract as the BEiT-3 artifact
(.claude/plans/kaggle-offline-embed.plan.md Section 3), so nothing downstream needs to
special-case which encoder produced a given embed_out/ directory.

Generated by notebooks/kaggle_embed_siglip2.ipynb (%%writefile) -- not tracked in git.
"""
from __future__ import annotations

import argparse
import json
import os
import re
import sys
import time
from pathlib import Path

import numpy as np
from PIL import Image

_FILENAME_RE = re.compile(r"^(L\d+_V\d+)_(\d+)\.(jpg|jpeg|png)$", re.IGNORECASE)
_NORM_TOLERANCE = 1e-3


def _parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--kf-root", type=Path, required=True)
    parser.add_argument("--out-dir", type=Path, required=True)
    parser.add_argument("--repo-dir", type=Path, required=True)
    parser.add_argument("--hf-repo-id", required=True)
    parser.add_argument("--hf-revision", required=True)
    parser.add_argument("--model-id", required=True)
    parser.add_argument("--chunk-size", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--preprocess-workers", type=int, default=8)
    parser.add_argument("--device", default="cuda")
    parser.add_argument(
        "--limit-videos", type=int, default=0, help="0 = khong gioi han; >0 chi cho smoke test"
    )
    parser.add_argument(
        "--limit-keyframes",
        type=int,
        default=0,
        help="0 = khong gioi han; >0 dung SAU khi vuot nguong nay (don vi video, khong cat "
        "giua video) -- dung cho smoke test theo so keyframe thay vi so video",
    )
    return parser.parse_args()


def discover_keyframes(kf_root: Path) -> dict[str, list[tuple[int, Path]]]:
    """Parse, validate, and group keyframe files. Raises on any parse/duplicate error."""
    image_paths = sorted(
        p for p in kf_root.rglob("*") if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
    )
    if not image_paths:
        raise FileNotFoundError(f"Khong tim thay anh keyframe nao duoi {kf_root}")

    errors: list[str] = []
    seen: dict[tuple[str, int], Path] = {}
    grouped: dict[str, list[tuple[int, Path]]] = {}

    for path in image_paths:
        match = _FILENAME_RE.match(path.name)
        if not match:
            errors.append(f"ten file khong khop pattern: {path}")
            continue
        video_id, frame_idx_str, _ext = match.groups()
        if path.parent.name != video_id:
            errors.append(f"video_id ({video_id}) khong khop thu muc cha ({path.parent.name}): {path}")
            continue
        frame_idx = int(frame_idx_str)
        key = (video_id, frame_idx)
        if key in seen:
            errors.append(f"trung (video_id, frame_idx)={key}: {path} va {seen[key]}")
            continue
        seen[key] = path
        grouped.setdefault(video_id, []).append((frame_idx, path))

    if errors:
        for line in errors[:50]:
            print(" -", line, file=sys.stderr)
        raise ValueError(f"{len(errors)} loi kiem ke bo keyframe -- dung lai truoc khi embed.")

    for video_id in grouped:
        grouped[video_id].sort(key=lambda item: item[0])  # frame_idx tang dan -> giu tau < t
    return grouped


def _select_video_ids(
    video_ids: list[str],
    grouped: dict[str, list[tuple[int, Path]]],
    limit_videos: int,
    limit_keyframes: int,
) -> list[str]:
    if limit_videos > 0:
        video_ids = video_ids[:limit_videos]
    if limit_keyframes > 0:
        selected: list[str] = []
        total = 0
        for video_id in video_ids:
            selected.append(video_id)
            total += len(grouped[video_id])
            if total >= limit_keyframes:
                break
        video_ids = selected
    return video_ids


def _is_video_done(out_dir: Path, video_id: str, expected_rows: int) -> bool:
    npy_path = out_dir / "vectors" / f"{video_id}.npy"
    jsonl_path = out_dir / "vectors" / f"{video_id}.jsonl"
    if not (npy_path.exists() and jsonl_path.exists()):
        return False
    with jsonl_path.open(encoding="utf-8") as f:
        num_lines = sum(1 for _ in f)
    if num_lines != expected_rows:
        return False
    vectors = np.load(npy_path)
    return vectors.shape[0] == expected_rows


def _embed_video(encoder, frame_records: list[tuple[int, Path]], chunk_size: int) -> np.ndarray:
    chunks = [frame_records[i : i + chunk_size] for i in range(0, len(frame_records), chunk_size)]
    vector_chunks = []
    for chunk in chunks:
        images = [Image.open(path) for _, path in chunk]
        try:
            vector_chunks.append(encoder.encode_image(images))
        finally:
            for image in images:
                image.close()
    return np.concatenate(vector_chunks, axis=0).astype(np.float32)


def _assert_normalized(vectors: np.ndarray, video_id: str) -> None:
    if vectors.shape[0] == 0:
        return
    sample_size = min(8, vectors.shape[0])
    sample_idx = np.random.default_rng(0).choice(vectors.shape[0], size=sample_size, replace=False)
    norms = np.linalg.norm(vectors[sample_idx], axis=1)
    bad = norms[np.abs(norms - 1.0) > _NORM_TOLERANCE]
    if bad.size:
        raise AssertionError(
            f"{video_id}: {bad.size} vector co norm ngoai "
            f"[1-{_NORM_TOLERANCE}, 1+{_NORM_TOLERANCE}]: {bad}"
        )


def _write_video_output(
    out_dir: Path, video_id: str, frame_records: list[tuple[int, Path]], vectors: np.ndarray
) -> None:
    vectors_dir = out_dir / "vectors"
    vectors_dir.mkdir(parents=True, exist_ok=True)

    jsonl_tmp = vectors_dir / f"{video_id}.jsonl.tmp"
    with jsonl_tmp.open("w", encoding="utf-8") as f:
        for frame_idx, _path in frame_records:
            keyframe_id = f"{video_id}_f{frame_idx}"
            record = {"keyframe_id": keyframe_id, "video_id": video_id, "frame_idx": frame_idx}
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    # Same np.save()-auto-suffix trap as the BEiT-3 script -- open the temp file as a file
    # object first so numpy never gets a chance to silently rewrite the ".tmp" name.
    npy_tmp = vectors_dir / f"{video_id}.npy.tmp"
    with npy_tmp.open("wb") as f:
        np.save(f, vectors)

    os.replace(npy_tmp, vectors_dir / f"{video_id}.npy")
    os.replace(jsonl_tmp, vectors_dir / f"{video_id}.jsonl")


def _cleanup_stale_tmp_files(vectors_dir: Path) -> None:
    if not vectors_dir.exists():
        return
    stale = sorted(vectors_dir.glob("*.tmp")) + sorted(vectors_dir.glob("*.tmp.npy"))
    for path in stale:
        path.unlink()
    if stale:
        print(f"Don {len(stale)} file tam con sot tu lan chay truoc: {[p.name for p in stale]}")


def main() -> None:
    args = _parse_args()
    # append, KHONG insert(0, ...): repo co package datasets/ trung ten thu vien HF
    # `datasets` -- insert o dau sys.path se khien `import datasets` (neu bat ky thu vien
    # nao trong chuoi phu thuoc cua transformers goi toi) bat nham package cua repo.
    sys.path.append(str(args.repo_dir))
    from app.encoders.siglip2 import SigLip2Encoder  # noqa: E402 (needs sys.path append above)

    grouped = discover_keyframes(args.kf_root)
    video_ids = _select_video_ids(sorted(grouped), grouped, args.limit_videos, args.limit_keyframes)

    _cleanup_stale_tmp_files(args.out_dir / "vectors")

    encoder = SigLip2Encoder(
        hf_repo_id=args.hf_repo_id,
        hf_revision=args.hf_revision,
        device=args.device,
        batch_size=args.batch_size,
        model_id=args.model_id,
        preprocess_workers=args.preprocess_workers,
    )

    total_videos = len(video_ids)
    total_vectors_written = 0
    start_time = time.monotonic()

    for i, video_id in enumerate(video_ids, start=1):
        frame_records = grouped[video_id]
        if _is_video_done(args.out_dir, video_id, len(frame_records)):
            print(f"[{i}/{total_videos}] {video_id}: da xong ({len(frame_records)} kf) -- bo qua (resume).")
            total_vectors_written += len(frame_records)
            continue

        video_start = time.monotonic()
        vectors = _embed_video(encoder, frame_records, args.chunk_size)
        if vectors.shape[0] != len(frame_records):
            raise AssertionError(
                f"{video_id}: so vector ({vectors.shape[0]}) != so keyframe ({len(frame_records)})"
            )
        _assert_normalized(vectors, video_id)
        _write_video_output(args.out_dir, video_id, frame_records, vectors)

        elapsed = time.monotonic() - video_start
        total_vectors_written += vectors.shape[0]
        print(f"[{i}/{total_videos}] {video_id}: {vectors.shape[0]} kf trong {elapsed:.1f}s")

    total_elapsed = time.monotonic() - start_time
    rate = total_vectors_written / total_elapsed if total_elapsed > 0 else 0.0
    print(
        f"\nXong: {total_videos} video, {total_vectors_written} vector, "
        f"{total_elapsed:.1f}s ({total_elapsed / 60:.1f} phut), {rate:.1f} keyframe/s."
    )


if __name__ == "__main__":
    main()


## 7. Smoke test ~1.000 keyframe

Chạy TRƯỚC khi treo job full 873 video — cho số đo throughput thật (keyframe/s), thay vì ước lượng. Dùng `--limit-keyframes` (không phải `--limit-videos`) vì mật độ keyframe/video lệch rất nhiều giữa batch.

In [ ]:
import subprocess
import time

smoke_start = time.monotonic()
subprocess.run(
    [
        "python", "/kaggle/working/embed_siglip2.py",
        "--kf-root", str(KF_ROOT),
        "--out-dir", str(OUT_DIR),
        "--repo-dir", str(REPO_DIR),
        "--hf-repo-id", HF_REPO_ID,
        "--hf-revision", HF_REVISION,
        "--model-id", MODEL_ID,
        "--chunk-size", str(CHUNK_SIZE),
        "--batch-size", str(BATCH_SIZE),
        "--preprocess-workers", str(PREPROCESS_WORKERS),
        "--device", "cuda",
        "--limit-keyframes", "1000",
    ],
    check=True, cwd=str(REPO_DIR),
)
smoke_elapsed = time.monotonic() - smoke_start
print(f"\nSmoke test: {smoke_elapsed:.1f}s")


In [ ]:
import numpy as np

smoke_npy_files = sorted(VECTORS_DIR.glob("*.npy"))
smoke_num_keyframes = int(sum(np.load(p).shape[0] for p in smoke_npy_files))
smoke_rate = smoke_num_keyframes / smoke_elapsed if smoke_elapsed > 0 else 0.0

first_video_id = smoke_npy_files[0].stem
first_vectors = np.load(smoke_npy_files[0])
norms = np.linalg.norm(first_vectors, axis=1)
print(f"{first_video_id}: shape={first_vectors.shape} norm min={norms.min():.5f} max={norms.max():.5f}")
assert abs(norms.min() - 1.0) < 1e-3 and abs(norms.max() - 1.0) < 1e-3, (
    "norm ngoai nguong -- dung lai, khong chay full 873 video truoc khi tim ra nguyen nhan"
)
assert first_vectors.shape[1] == EMBEDDING_DIM, (
    f"embedding_dim thuc te {first_vectors.shape[1]} != ky vong {EMBEDDING_DIM}"
)

print(f"\nSmoke test: {smoke_num_keyframes} keyframe trong {smoke_elapsed:.1f}s -> {smoke_rate:.1f} keyframe/s")
est_total_seconds = EXPECTED_NUM_KEYFRAMES / smoke_rate if smoke_rate > 0 else float("inf")
print(
    f"Uoc luong thoi gian full {EXPECTED_NUM_KEYFRAMES} keyframe (tuyen tinh theo mau smoke): "
    f"~{est_total_seconds / 3600:.1f} gio -- uoc luong tho, mat do kf lech nhieu giua cac "
    "batch, mau smoke co the khong dai dien cho toan corpus."
)
print(
    "\nNeu GPU van ranh nhieu trong luc chay (xem !nvidia-smi o mot cell khac trong luc "
    "job dang chay), thu tang PREPROCESS_WORKERS truoc, BATCH_SIZE sau -- xem comment o "
    "cell cau hinh."
)


## 8. Chạy full 873 video — cell nặng nhất

Chỉ bấm sau khi mục 5 (kiểm kê) và mục 7 (smoke test) đều xanh và bạn đã đọc số `keyframe/s` + giờ ước tính. Kaggle giới hạn 30 giờ GPU/tuần — kiểm tra ngân sách còn lại trước khi bấm nếu định chạy cả candidate A2 (PE-Core) trong cùng tuần. Nếu session bị cắt giữa chừng, script tự resume ở lần chạy sau (bỏ qua video đã có đủ `.npy`/`.jsonl`).

In [ ]:
full_run_start = time.monotonic()
subprocess.run(
    [
        "python", "/kaggle/working/embed_siglip2.py",
        "--kf-root", str(KF_ROOT),
        "--out-dir", str(OUT_DIR),
        "--repo-dir", str(REPO_DIR),
        "--hf-repo-id", HF_REPO_ID,
        "--hf-revision", HF_REVISION,
        "--model-id", MODEL_ID,
        "--chunk-size", str(CHUNK_SIZE),
        "--batch-size", str(BATCH_SIZE),
        "--preprocess-workers", str(PREPROCESS_WORKERS),
        "--device", "cuda",
        "--limit-videos", "0",
        "--limit-keyframes", "0",
    ],
    check=True, cwd=str(REPO_DIR),
)
full_run_elapsed = time.monotonic() - full_run_start
print(f"\nChay full: {full_run_elapsed:.1f}s ({full_run_elapsed / 3600:.2f} gio)")


## 9. Kiểm tra output

Fail loud nếu còn rác `.tmp`/tên lạ, hoặc số hàng `.npy` lệch số dòng `.jsonl`, hoặc `embedding_dim` không phải 1152.

In [ ]:
import re

_VIDEO_ID_RE = re.compile(r"^L\d+_V\d+$")

all_npy = sorted(VECTORS_DIR.glob("*.npy"))
npy_files = [p for p in all_npy if _VIDEO_ID_RE.match(p.stem)]
garbage = [p for p in all_npy if p not in npy_files]
if garbage:
    print(f"Rac con sot ({len(garbage)} file, ten khong khop video_id) -- nen xoa tay:")
    for p in garbage:
        print("  -", p.name)
    raise ValueError(
        f"{len(garbage)} file rac trong {VECTORS_DIR} -- xoa thu cong cac file tren roi chay "
        "lai cell nay (khong tu xoa o day de tranh xoa nham)."
    )

print(f"So video co output: {len(npy_files)} (ky vong {EXPECTED_NUM_VIDEOS})")

total_vectors = 0
mismatches = []
dim_mismatches = []
for npy_path in npy_files:
    vectors = np.load(npy_path)
    if vectors.shape[1] != EMBEDDING_DIM:
        dim_mismatches.append((npy_path.stem, vectors.shape[1]))
    jsonl_path = VECTORS_DIR / f"{npy_path.stem}.jsonl"
    with jsonl_path.open(encoding="utf-8") as f:
        num_lines = sum(1 for _ in f)
    if vectors.shape[0] != num_lines:
        mismatches.append((npy_path.stem, vectors.shape[0], num_lines))
    total_vectors += vectors.shape[0]

print(f"Tong vector: {total_vectors} (ky vong {EXPECTED_NUM_KEYFRAMES})")
if mismatches:
    raise AssertionError(f"{len(mismatches)} video lech so hang .npy / so dong .jsonl: {mismatches[:10]}")
if dim_mismatches:
    raise AssertionError(f"{len(dim_mismatches)} video co embedding_dim sai: {dim_mismatches[:10]}")
print(f"So hang .npy khop so dong .jsonl, embedding_dim == {EMBEDDING_DIM} cho toan bo video: OK")

!du -sh {OUT_DIR}


## 10. Image probe + text probe (manifest v2)

Local **giờ đã có ảnh** (khác notebook BEiT-3 lúc trước) — nên probe được cả bằng ảnh, không
chỉ text. Image probe: chọn 16 `keyframe_id` cố định, trải đều theo chỉ số (không phải 16
cái đầu — tránh dồn hết vào batch L21) để phủ nhiều batch, rồi trích thẳng vector từ
`vectors/*.npy` đã ghi (không encode lại). Text probe: encode lại 20 câu cố định, giống hệt
`kaggle_embed_keyframes.ipynb` mục 12.

In [ ]:
import json

import numpy as np

all_records = []  # (keyframe_id, video_id, frame_idx, row_index)
for jsonl_path in sorted(VECTORS_DIR.glob("*.jsonl")):
    video_id = jsonl_path.stem
    with jsonl_path.open(encoding="utf-8") as f:
        for row_index, line in enumerate(f):
            record = json.loads(line)
            all_records.append((record["keyframe_id"], video_id, record["frame_idx"], row_index))

if len(all_records) < IMAGE_PROBE_COUNT:
    raise ValueError(f"Chi co {len(all_records)} keyframe, khong du {IMAGE_PROBE_COUNT} cho image probe.")

# Trai deu theo chi so thay vi lay 16 cai dau (deu nam trong batch L21) -- de probe phu
# nhieu batch khac nhau.
step = len(all_records) // IMAGE_PROBE_COUNT
probe_records = [all_records[i * step] for i in range(IMAGE_PROBE_COUNT)]

probe_vectors = np.empty((IMAGE_PROBE_COUNT, EMBEDDING_DIM), dtype=np.float32)
probe_meta = []
for i, (keyframe_id, video_id, frame_idx, row_index) in enumerate(probe_records):
    vectors = np.load(VECTORS_DIR / f"{video_id}.npy")
    probe_vectors[i] = vectors[row_index]
    probe_meta.append({"keyframe_id": keyframe_id, "video_id": video_id, "frame_idx": frame_idx})

PROBE_DIR.mkdir(parents=True, exist_ok=True)
np.save(PROBE_DIR / "image_probe_vectors.npy", probe_vectors)
(PROBE_DIR / "image_probe.json").write_text(
    json.dumps(probe_meta, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Image probe: {IMAGE_PROBE_COUNT} keyframe, luu tai {PROBE_DIR}")
for m in probe_meta:
    print(" -", m["keyframe_id"])


In [ ]:
%%writefile /kaggle/working/probe_encode_siglip2.py
"""Encode fixed probe strings with SigLip2Encoder -- parity-check artifact for local import.

Generated by notebooks/kaggle_embed_siglip2.ipynb (%%writefile) -- not tracked in git.
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np


def _parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo-dir", type=Path, required=True)
    parser.add_argument("--hf-repo-id", required=True)
    parser.add_argument("--hf-revision", required=True)
    parser.add_argument("--model-id", required=True)
    parser.add_argument("--probe-texts-path", type=Path, required=True)
    parser.add_argument("--out-npy-path", type=Path, required=True)
    parser.add_argument("--device", default="cuda")
    return parser.parse_args()


def main() -> None:
    args = _parse_args()
    sys.path.append(str(args.repo_dir))
    from app.encoders.siglip2 import SigLip2Encoder  # noqa: E402

    probe_texts = json.loads(args.probe_texts_path.read_text(encoding="utf-8"))
    encoder = SigLip2Encoder(
        hf_repo_id=args.hf_repo_id,
        hf_revision=args.hf_revision,
        device=args.device,
        batch_size=len(probe_texts),
        model_id=args.model_id,
    )
    vectors = encoder.encode_text(probe_texts)
    np.save(args.out_npy_path, vectors.astype(np.float32))
    print(f"Da encode {len(probe_texts)} probe text -> {args.out_npy_path}, shape={vectors.shape}")


if __name__ == "__main__":
    main()


In [ ]:
PROBE_DIR.mkdir(parents=True, exist_ok=True)
(PROBE_DIR / "probe_texts.json").write_text(
    json.dumps(PROBE_TEXTS, ensure_ascii=False, indent=2), encoding="utf-8"
)

subprocess.run(
    [
        "python", "/kaggle/working/probe_encode_siglip2.py",
        "--repo-dir", str(REPO_DIR),
        "--hf-repo-id", HF_REPO_ID,
        "--hf-revision", HF_REVISION,
        "--model-id", MODEL_ID,
        "--probe-texts-path", str(PROBE_DIR / "probe_texts.json"),
        "--out-npy-path", str(PROBE_DIR / "probe_text_vectors.npy"),
        "--device", "cuda",
    ],
    check=True, cwd=str(REPO_DIR),
)


## 11. Sinh `manifest.json` (v2)

So với artifact BEiT-3 (manifest v1, hash file checkpoint local), artifact này không có file
checkpoint để hash — thay bằng `hf_revision` (commit SHA ghim cứng) + `weight_files_sha256`
(hash trực tiếp file `.safetensors` mà `from_pretrained` thực sự tải/dùng, lấy qua
`hf_hub_download` — cùng file, không đoán đường dẫn cache). Lệch bất kỳ field nào ở đây khi
so với local phải **chặn import**, không chỉ cảnh báo (plan §T4).

In [ ]:
import hashlib
import subprocess

from huggingface_hub import hf_hub_download


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


git_sha = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), capture_output=True, text=True, check=True
).stdout.strip()

import torch
import transformers

weight_path = Path(
    hf_hub_download(repo_id=HF_REPO_ID, filename="model.safetensors", revision=HF_REVISION)
)
weight_sha256 = _sha256(weight_path)

final_npy_files = sorted(VECTORS_DIR.glob("*.npy"))
final_num_videos = len(final_npy_files)
final_num_keyframes = int(sum(np.load(p).shape[0] for p in final_npy_files))

manifest = {
    "manifest_version": 2,
    "encoder_kind": "siglip2",
    "model_id": MODEL_ID,
    "embedding_dim": EMBEDDING_DIM,
    "hf_repo_id": HF_REPO_ID,
    "hf_revision": HF_REVISION,
    "weight_files_sha256": {"model.safetensors": weight_sha256},
    "preprocess": (
        "image: Siglip2ImageProcessor mac dinh cua checkpoint (resize 384x384, patch 14, "
        "normalize theo preprocessor_config.json) | "
        "text: padding=max_length, max_length=64, truncation=True (BAT BUOC, khong phai "
        "dynamic padding -- huggingface.co/docs/transformers/main/en/model_doc/siglip2)"
    ),
    "library_versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
    },
    "num_videos": final_num_videos,
    "num_keyframes": final_num_keyframes,
    "git_sha": git_sha,
    "probe_text_count": len(PROBE_TEXTS),
    "image_probe": {
        "count": IMAGE_PROBE_COUNT,
        "keyframe_ids": [m["keyframe_id"] for m in probe_meta],
    },
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))

if final_num_videos != EXPECTED_NUM_VIDEOS or final_num_keyframes != EXPECTED_NUM_KEYFRAMES:
    print(
        "\nCANH BAO: num_videos/num_keyframes trong manifest KHONG khop ky vong "
        f"({EXPECTED_NUM_VIDEOS}/{EXPECTED_NUM_KEYFRAMES})."
    )


## 12. Mang dữ liệu về local + resume nếu session bị cắt

### Mang về local (khi mục 8 đã báo đủ 873/873 video)

1. Commit notebook version → tab **Output** → **Download All**
   (`embed_out_siglip2/` — chỉ vector, không kèm ảnh, tương tự artifact BEiT-3).
2. Đặt vào `data/embed_out_siglip2/` ở local.
3. Chờ `scripts/import_embed_out.py --encoder siglip2` (T5 của
   `.claude/plans/encoder-swap-siglip2-pecore.plan.md` — **CHƯA implement** tại thời điểm
   viết notebook này) — nó sẽ chạy cổng parity (text probe + image probe + hash weight)
   TRƯỚC khi ghi bất kỳ vector nào vào Milvus, và ghi vào collection
   `keyframes_siglip2_so400m_p14_384` — **không đụng** collection BEiT-3 hay MongoDB.

### Nếu Kaggle cắt session giữa chừng, hoặc cần chạy tiếp phiên sau

1. Commit notebook version hiện tại (dù chạy dở) để `/kaggle/working/embed_out_siglip2/`
   (phần đã xong) trở thành Output.
2. Mở session mới, chạy lại mục 1–6 (GPU/clone/dependency/config/kiểm kê/sinh script) —
   các bước này idempotent.
3. **Add** chính Output của lần chạy trước làm Kaggle Dataset input mới, rồi copy nó đè lên
   `OUT_DIR` trước khi chạy lại mục 8:
   ```python
   import shutil
   PREVIOUS_SESSION_DATASET_DIR_NAME = ""  # ví dụ: "aic-embed-out-siglip2-session1"
   if PREVIOUS_SESSION_DATASET_DIR_NAME:
       prev_dir = Path(f"/kaggle/input/{PREVIOUS_SESSION_DATASET_DIR_NAME}")
       shutil.copytree(prev_dir, OUT_DIR, dirs_exist_ok=True)
   ```
4. Chạy lại đúng cell ở mục 8 — video đã có đủ `.npy`/`.jsonl` khớp số dòng được bỏ qua tự
   động. Lặp lại tới khi mục 9 báo đủ 873/873.
5. Mục 10–11 (probe + manifest) chỉ cần chạy **một lần cuối cùng**, sau khi mục 9 đã xác
   nhận đủ 873/873 — không cần chạy lại mỗi phiên.